# Double Machine Learning: PISA 2022 Germany

Treatment: `PRESCHOOL`  
Outcome: `READ`  
Covariates: `GENDER`, `AGE`, `IMMIG`, `HISCED`, `HOMEPOS`, `ESCS`, `CREATHME`, `PA188Q03JA`, `PA005Q01TA`, `HISEI`

In [ ]:
# Import packages

import pandas as pd
import numpy as np
import warnings

from sklearn.model_selection import cross_val_predict, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import (
    LinearRegression, LassoCV, RidgeCV, ElasticNetCV, LogisticRegressionCV
)
from sklearn.ensemble import (
    RandomForestRegressor, RandomForestClassifier,
    GradientBoostingRegressor, GradientBoostingClassifier
)
from sklearn.neural_network import MLPRegressor, MLPClassifier

import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.simplefilter("ignore")
np.random.seed(1234)

## 1. Load data and define variables

In [ ]:
# Load data

file = "PISA2022deu.csv"
data_raw = pd.read_csv(file)

# Recode treatment

data_raw["PREEDU"] = pd.to_numeric(data_raw["PREEDU"], errors="coerce")
data_raw["PRESCHOOL"] = 1 - data_raw["PREEDU"]

# Define variables

D_var = "PRESCHOOL"
Y_var = "READ"

controls = [
    "GENDER", "AGE", "IMMIG", "HISCED", "HOMEPOS", "ESCS",
    "CREATHME", "PA188Q03JA", "PA005Q01TA", "HISEI"
]

selected_vars = [Y_var, D_var] + controls

# Keep complete cases

usedata = data_raw[selected_vars].copy()

print("Original sample size:", usedata.shape[0])
usedata = usedata.dropna().reset_index(drop=True)
print("Complete-case sample size:", usedata.shape[0])

usedata[D_var].value_counts(dropna=False).sort_index()

In [ ]:
# Check treatment coding

unique_d = sorted(usedata[D_var].dropna().unique())
print("Observed PRESCHOOL categories:", unique_d)

if len(unique_d) != 2:
    raise ValueError(f"PRESCHOOL must be dichotomous. Observed categories: {unique_d}")

if set(unique_d) != {0, 1}:
    mapping = {unique_d[0]: 0, unique_d[1]: 1}
    print("Recode PRESCHOOL using mapping:", mapping)
    usedata[D_var] = usedata[D_var].map(mapping)
else:
    print("PRESCHOOL is already coded as 0/1.")

usedata[D_var].value_counts().sort_index()

## 2. Prepare covariates

In [ ]:
# Define covariate groups

categorical_controls = ["GENDER", "IMMIG", "HISCED", "PA188Q03JA", "PA005Q01TA"]
continuous_controls = ["AGE", "HOMEPOS", "ESCS", "CREATHME", "HISEI"]

basic_controls = ["GENDER", "AGE", "IMMIG", "HISCED", "HOMEPOS", "ESCS", "HISEI"]
all_controls = controls.copy()

# Build numeric design matrix

def make_design_matrix(df, covariates):
    X = df[covariates].copy()
    cats = [c for c in categorical_controls if c in covariates]
    for c in cats:
        X[c] = X[c].astype("category")
    X = pd.get_dummies(X, columns=cats, drop_first=True, dtype=float)
    return X.astype(float)

Y = usedata[Y_var].astype(float).reset_index(drop=True)
D = usedata[D_var].astype(float).reset_index(drop=True)

Z_basic = make_design_matrix(usedata, basic_controls).reset_index(drop=True)
Z_all = make_design_matrix(usedata, all_controls).reset_index(drop=True)
Z_const = pd.DataFrame({"Const": np.ones(len(usedata))})

print("Z_basic shape:", Z_basic.shape)
print("Z_all shape:", Z_all.shape)

Z_all.head()

## 3. OLS benchmarks

In [ ]:
# Estimate OLS benchmarks

def fit_ols(Y, D, Z=None, *, name):
    if Z is None:
        X = pd.DataFrame({D_var: D})
    else:
        X = pd.concat([pd.Series(D, name=D_var).reset_index(drop=True), Z.reset_index(drop=True)], axis=1)
    X = sm.add_constant(X, has_constant="add")
    lm = sm.OLS(Y, X).fit(cov_type="HC3")
    return pd.DataFrame({
        "estimate": lm.params[D_var],
        "stderr": lm.bse[D_var],
        "lower": lm.params[D_var] - 1.96 * lm.bse[D_var],
        "upper": lm.params[D_var] + 1.96 * lm.bse[D_var]
    }, index=[name])

ols_table = pd.concat([
    fit_ols(Y, D, None, name="OLS No Controls"),
    fit_ols(Y, D, Z_basic, name="OLS Basic Controls"),
    fit_ols(Y, D, Z_all, name="OLS All Controls")
])

ols_table

## 4. DML functions

In [ ]:
# Define DML estimator

def dml(X, D, y, modely, modeld, *, nfolds=5, classifier=False):
    X = X.reset_index(drop=True)
    D = pd.Series(D).reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)

    cv = KFold(n_splits=nfolds, shuffle=True, random_state=123)

    yhat = cross_val_predict(modely, X, y, cv=cv, n_jobs=-1)

    if classifier:
        Dhat = cross_val_predict(modeld, X, D, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    else:
        Dhat = cross_val_predict(modeld, X, D, cv=cv, n_jobs=-1)

    resy = y - yhat
    resD = D - Dhat

    dml_data = pd.DataFrame({"resy": resy, "resD": resD})
    ols_mod = smf.ols(formula="resy ~ 1 + resD", data=dml_data).fit(cov_type="HC3")

    point = ols_mod.params["resD"]
    stderr = ols_mod.bse["resD"]
    epsilon = ols_mod.resid

    return point, stderr, yhat, Dhat, resy, resD, epsilon


# Summarize DML results

def summary(point, stderr, yhat, Dhat, resy, resD, epsilon, X, D, y, *, name):
    return pd.DataFrame({
        "estimate": point,
        "stderr": stderr,
        "lower": point - 1.96 * stderr,
        "upper": point + 1.96 * stderr,
        "rmse y": np.sqrt(np.mean(np.asarray(resy) ** 2)),
        "rmse D": np.sqrt(np.mean(np.asarray(resD) ** 2))
    }, index=[name])

## 5. DML with linear first-stage models

In [ ]:
# Run DML benchmarks

modely = make_pipeline(StandardScaler(), LinearRegression())
modeld = make_pipeline(StandardScaler(), LinearRegression())

result_OLS = dml(Z_const, D, Y, modely, modeld, nfolds=5, classifier=False)
table = summary(*result_OLS, Z_const, D, Y, name="OLS No Controls")

result_basic = dml(Z_basic, D, Y, modely, modeld, nfolds=5, classifier=False)
table = pd.concat([table, summary(*result_basic, Z_basic, D, Y, name="OLS Basic Controls")])

result_all = dml(Z_all, D, Y, modely, modeld, nfolds=5, classifier=False)
table = pd.concat([table, summary(*result_all, Z_all, D, Y, name="OLS All Controls")])

table

## 6. DML with machine-learning first-stage models

In [ ]:
# Define first-stage models

cv_inner = KFold(n_splits=10, shuffle=True, random_state=123)

models = {
    "LassoCV / Logit-L1": (
        make_pipeline(StandardScaler(), LassoCV(cv=cv_inner, random_state=123, max_iter=10000)),
        make_pipeline(StandardScaler(), LogisticRegressionCV(
            Cs=10, cv=cv_inner, penalty="l1", solver="saga", scoring="neg_log_loss",
            max_iter=5000, random_state=123
        )),
        True
    ),
    "RidgeCV / Logit-L2": (
        make_pipeline(StandardScaler(), RidgeCV(cv=cv_inner)),
        make_pipeline(StandardScaler(), LogisticRegressionCV(
            Cs=10, cv=cv_inner, penalty="l2", solver="lbfgs", scoring="neg_log_loss",
            max_iter=5000
        )),
        True
    ),
    "ENetCV / Logit-ENet": (
        make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=0.5, cv=cv_inner, random_state=123, max_iter=10000)),
        make_pipeline(StandardScaler(), LogisticRegressionCV(
            Cs=10, cv=cv_inner, penalty="elasticnet", solver="saga", l1_ratios=[0.5],
            scoring="neg_log_loss", max_iter=5000, random_state=123
        )),
        True
    ),
    "RF": (
        RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=123, n_jobs=-1),
        RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=123, n_jobs=-1),
        True
    ),
    "Boosted Trees": (
        GradientBoostingRegressor(max_depth=3, n_iter_no_change=5, random_state=123),
        GradientBoostingClassifier(max_depth=3, n_iter_no_change=5, random_state=123),
        True
    ),
    "NN (Early Stopping)": (
        make_pipeline(StandardScaler(), MLPRegressor(
            hidden_layer_sizes=(50, 50), activation="relu", solver="adam",
            alpha=0.0001, batch_size=200, learning_rate_init=0.001,
            max_iter=500, shuffle=True, random_state=123, early_stopping=True,
            validation_fraction=0.2, n_iter_no_change=10
        )),
        make_pipeline(StandardScaler(), MLPClassifier(
            hidden_layer_sizes=(50, 50), activation="relu", solver="adam",
            alpha=0.0001, batch_size=200, learning_rate_init=0.001,
            max_iter=500, shuffle=True, random_state=123, early_stopping=True,
            validation_fraction=0.2, n_iter_no_change=10
        )),
        True
    )
}

In [ ]:
# Run DML models

results = {
    "OLS No Controls": result_OLS,
    "OLS Basic Controls": result_basic,
    "OLS All Controls": result_all
}

for name, (modely, modeld, is_classifier) in models.items():
    print("Running", name)
    result = dml(Z_all, D, Y, modely, modeld, nfolds=5, classifier=is_classifier)
    results[name] = result
    table = pd.concat([table, summary(*result, Z_all, D, Y, name=name)])

table

## 7. First-stage prediction performance

In [ ]:
# Compare first-stage RMSEs

rmses = table[["rmse y", "rmse D"]]
rmses

In [ ]:
# Select best model by RMSE rank

print("Lowest RMSE y:", rmses["rmse y"].idxmin())
print("Lowest RMSE D:", rmses["rmse D"].idxmin())

rank_table = rmses.rank(axis=0)
rank_table["avg_rank"] = rank_table.mean(axis=1)
best_name = rank_table["avg_rank"].idxmin()

print("Best average RMSE rank:", best_name)

rank_table.sort_values("avg_rank")

## 8. Least Squares Model Averaging

In [ ]:
# Build prediction matrices

model_order = list(results.keys())

yhat_all = pd.concat(
    [pd.Series(results[name][2], name=name) for name in model_order],
    axis=1
)

Dhat_all = pd.concat(
    [pd.Series(results[name][3], name=name) for name in model_order],
    axis=1
)

# Estimate model-average residuals

ma_y = sm.OLS(Y.reset_index(drop=True), yhat_all).fit()
ma_d = sm.OLS(D.reset_index(drop=True), Dhat_all).fit()

weights_y = ma_y.params.rename("weight_y")
weights_d = ma_d.params.rename("weight_D")
weights = pd.concat([weights_y, weights_d], axis=1)

# Estimate final treatment effect

lm_ma_data = pd.DataFrame({"resy": ma_y.resid, "resD": ma_d.resid})
lm_ma = smf.ols("resy ~ 1 + resD", data=lm_ma_data).fit(cov_type="HC3")

lsma = pd.DataFrame({
    "estimate": lm_ma.params["resD"],
    "stderr": lm_ma.bse["resD"]
}, index=["Least Squares Model Average"])

lsma["lower"] = lsma["estimate"] - 1.96 * lsma["stderr"]
lsma["upper"] = lsma["estimate"] + 1.96 * lsma["stderr"]

final_table = pd.concat([table[["estimate", "stderr", "lower", "upper"]], lsma], axis=0)

final_table

In [ ]:
# Inspect model-averaging weights

weights

## 9. Output tables

In [ ]:
# Export results

ols_table.to_csv("dml_ols_table_preschool_read.csv")
table.to_csv("dml_results_preschool_read.csv")
final_table.to_csv("dml_final_table_preschool_read.csv")
weights.to_csv("dml_model_averaging_weights_preschool_read.csv")

print("Saved output tables.")